# DevBrain – AI Memory for Your Codebase

A RAG-powered assistant that helps developers understand their codebase, decisions, and history.

## Features
- Ask questions about your codebase
- Retrieve relevant files and explanations
- Uses embeddings + vector search (Chroma)
- Supports Hugging Face LLMs
- Interactive chat UI with Gradio

## Example Questions
- "Where is authentication handled?"
- "What does this service do?"
- "Why is this function slow?"

In [2]:
import os
from dotenv import load_dotenv
from pathlib import Path

# LangChain + RAG
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

# Hugging Face
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langchain_community.embeddings import HuggingFaceEmbeddings

# UI
import gradio as gr

# Utils
import shutil

In [4]:
load_dotenv(override=True)

hf_token = os.getenv("HG_TOKEN")

if hf_token and hf_token.startswith("hf_"):
    print("Hugging Face token loaded.")
else:
    print("No valid HF token found.")

Hugging Face token loaded.


In [5]:
def load_codebase(path: str):
    documents = []
    path = Path(path)

    for file in path.rglob("*"):
        if file.suffix in [".js", ".ts", ".jsx", ".tsx", ".json", ".md"]:
            try:
                content = file.read_text(encoding="utf-8")
                documents.append(
                    Document(
                        page_content=content,
                        metadata={"source": str(file)}
                    )
                )
            except Exception:
                continue

    print(f"Loaded {len(documents)} files")
    return documents

In [6]:
def split_documents(documents):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=800,
        chunk_overlap=200
    )
    chunks = splitter.split_documents(documents)
    print(f"Created {len(chunks)} chunks")
    return chunks

In [7]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded.")

/var/folders/13/c33dyrs50qsfzm5d1gcrplvr0000gn/T/ipykernel_3265/971521747.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


Embedding model loaded.


In [8]:
DB_PATH = "devbrain_db"

def create_vectorstore(chunks):
    if os.path.exists(DB_PATH):
        shutil.rmtree(DB_PATH)

    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_model,
        persist_directory=DB_PATH
    )

    print("Vectorstore created.")
    return vectorstore

In [ ]:
HF_MODEL = "meta-llama/Llama-3.1-8B"
tokenizer = AutoTokenizer.from_pretrained(HF_MODEL, use_auth_token=hf_token)
model = AutoModelForCausalLM.from_pretrained(HF_MODEL, use_auth_token=hf_token)

hf_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto",
    max_new_tokens=512
)

print(f"LLM loaded: {HF_MODEL}")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Device set to use mps:0


LLM loaded: TinyLlama/TinyLlama-1.1B-Chat-v1.0


In [11]:
SYSTEM_PROMPT = """
You are a senior software engineer assistant.

Answer questions about a codebase using the provided context.
Be precise, technical, and helpful.

If unsure, say you don't know.

Context:
{context}
"""

In [12]:
def answer_question(question, history, retriever):
    docs = retriever.invoke(question)

    context = "\n\n".join([
        f"{doc.metadata['source']}:\n{doc.page_content[:500]}"
        for doc in docs
    ])

    prompt = SYSTEM_PROMPT.format(context=context) + f"\n\nQuestion: {question}\nAnswer:"

    result = hf_pipeline(prompt)[0]["generated_text"]

    return result

In [13]:
def setup_rag(project_path):
    docs = load_codebase(project_path)
    chunks = split_documents(docs)
    vectorstore = create_vectorstore(chunks)
    retriever = vectorstore.as_retriever()

    return retriever

In [14]:
retriever_holder = {"retriever": None}

def initialize(project_path):
    retriever_holder["retriever"] = setup_rag(project_path)
    return "RAG system initialized!"

def chat_fn(message, history):
    retriever = retriever_holder["retriever"]
    if not retriever:
        return "Please initialize the system first."
    return answer_question(message, history, retriever)

with gr.Blocks() as ui:
    gr.Markdown("## DevBrain – AI Codebase Assistant")

    with gr.Row():
        project_path = gr.Textbox(label="Project Path", placeholder="/path/to/codebase")
        init_btn = gr.Button("Initialize")

    status = gr.Textbox(label="Status")

    chatbot = gr.ChatInterface(chat_fn)

    init_btn.click(
        fn=initialize,
        inputs=[project_path],
        outputs=[status]
    )

ui.launch()

/Users/josedev/Documents/research/andela/llm_engineering/.venv/lib/python3.12/site-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


## Future Improvements

- Add Git commit ingestion (history-aware RAG)
- Add error logs ingestion
- Show source files in UI
- Add re-ranking for better retrieval accuracy
- Streaming responses
- Multi-project memory
- Deploy as SaaS / internal dev tool